In [5]:
from datasets import load_from_disk

In [3]:
from transformers import AutoModelForTokenClassification
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_path)

model_path = "E:\\PROJECTS\\Privacy-Risk-Analyser\\models\\xlm-roberta-ner" 

model = AutoModelForTokenClassification.from_pretrained(model_path)
tokenizer = tokenizer.from_pretrained(model_path)


In [6]:
# Loading dataset
dataset_dict = load_from_disk("E:/PROJECTS/Privacy-Risk-Analyser/data/privacy_ner_dataset")

In [7]:
# Loading tokenizer and define label list
model_checkpoint = "xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

label_list = dataset_dict["train"].features["labels"].feature.names
label_to_id = {l: i for i, l in enumerate(label_list)}
num_labels = len(label_list)

In [9]:
# Test data
test_dataset = tokenized_datasets["test"]  


In [15]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
        padding="max_length",         # Ensure uniform length
        max_length=128                # Or 256 depending on your model input limit
    )

    labels = []
    for i, label in enumerate(examples["labels"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)  # Mask token
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx
        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs


In [17]:
from datasets import Dataset

# Assuming you have test_data = [{"tokens": [...], "labels": [...]}, ...]
test_dataset = Dataset.from_list(test_dataset)

# Apply tokenization
tokenized_test_dataset = test_dataset.map(tokenize_and_align_labels, batched=True)


Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [18]:
trainer = Trainer(model=model, tokenizer=tokenizer)
predictions = trainer.predict(tokenized_test_dataset)

# Get label predictions
pred_labels = predictions.predictions.argmax(-1)


C:\Users\Dell\AppData\Local\Temp\ipykernel_21212\442059172.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(model=model, tokenizer=tokenizer)
